# Creación de un *CRUD* con `sqlite3` (intermedio)
En esta versión del código, haré uso de funciones para simplificar las tareas del CRUD.

Recomendación: instalar extensión [SQLite Viewer](https://marketplace.visualstudio.com/items?itemName=qwtel.sqlite-viewer)

In [64]:
import sqlite3
from pathlib import Path
import os
import json

## Listar archivos de base de datos

In [55]:
os.getcwd()

'/home/daniel/code/my_python_course/sqlite_crud'

In [57]:
db_files = [i for i in Path("./databases/").iterdir() if i.suffix == ".db"]
print(db_files)

[PosixPath('databases/library.db')]


## Funciones auxiliares

Crearé algunas funciones auxiliares que me permitan realizar algunas tareas que ayuden en las tareas de mantenimiento de la base de datos y sus tablas.

### Listado de tablas de la BD
Esta función devuelve una lista de las tablas que contiene la BD de la conexión activa.

In [ ]:
def list_tables(conn = conn, cursor=cursor):
    tables = (cursor
              .execute("SELECT name FROM sqlite_master WHERE type='table'")
              .fetchall()
    )
    tables = [i[0] for i in tables if i[0] != "sqlite_sequence"]
    return tables

## Iniciar conexión  

In [98]:
def init_connection(db_file : str) -> tuple[sqlite3.Connection, sqlite3.Cursor] | None :
    # check if file exists
    if (Path("./databases/") / db_file).exists():
        try:
            conn = sqlite3.connect(db_file)
            cursor = conn.cursor()
            print("Connection created successfuly!")
            return conn, cursor
        except Exception as e:
            print(f"Error: {e}")
            return None
    else:
        print(f"{db_file} does not exist. I will create it.")
        try:
            conn = sqlite3.connect(Path("./databases/") / db_file)
            cursor = conn.cursor()
            print("Connection created successfuly!")
            print(f"Database file created: ./databases/{db_file}")
            return conn, cursor
        except Exception as e:
            print(f"Error: {e}")
            return None

In [119]:
conn, cursor = init_connection("inventario.db")

inventario.db does not exist. I will create it.
Connection created successfuly!
Database file created: ./databases/inventario.db


## Creación de tabla
Esta función crea una tabla a partir de una *query* con el código SQL de creación de la tabla.

In [ ]:
def create_table(table_name, query, conn, cursor):
    try:
        # check if table exists
        if not table_name in list_tables():
            cursor.execute(query)
            conn.commit()

            # get the database list
            db_details = conn.execute("PRAGMA database_list").fetchall()
            db_name = db_details[0][2].split("/")[-1]

            # success message
            print(f"Table {table_name} was successfuly created at database {db_name}!")
        
        # table was already created
        else:
            print(f"Table {table_name} already exists. No actions were taken.")
    except Exception as e:
        print(f"Error: {e}")

In [29]:
create_table_query = '''
create table if not exists books
(id integer primary key autoincrement,
title text unique,
author text,
publish_year integer,
available bool)
'''

In [120]:
create_table_productos_query ='''
    CREATE TABLE IF NOT EXISTS productos (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre          TEXT    NOT NULL,
        marca           TEXT    NOT NULL,
        categoria       TEXT    NOT NULL,
        precio          REAL    NOT NULL,
        stock           INTEGER NOT NULL,
        especificaciones TEXT,
        fecha_ingreso   TEXT
    )
'''

In [121]:
create_table("productos", create_table_productos_query, conn, cursor)

Table productos was successfuly created at database inventario.db!


## Inserción de datos
Insertamos registros en la base de datos

Inserción de registros usando *placeholders* (:column_name)

In [122]:
def populate_table(table_name, cols, values, data_structure="dict", conn=conn, cursor=cursor):
    
    # check if data_structure value is valid:
    if data_structure not in ["tuples", "dict"]:
        print("Invalid data_structure value: must be tuples or dict!")
        return None
    
    # create a dictionary containing column names as keys
    # and values contained in each tuple provided in values
    
    if data_structure != "dict":
        values = [{k:v for k,v in zip(cols, t)}
                  for t in values]
    
    # create placeholders
    placeholders = []
    for c in cols:
        placeholders.append(f":{c}")
    placeholders = ", ".join(placeholders)

    insert_query = f'''
    insert into {table_name} ({", ".join(cols)})
    values ({placeholders}) ;
    '''

    # insert records into db
    cursor.executemany(insert_query, values)
    conn.commit()
    
    # check if insertion was ok
    records_count_in_values = len(values)
    records_count_in_table= cursor.execute(f"select count(*) from {table_name} ;").fetchone()[0]

    if records_count_in_table == records_count_in_values:
        print(f"Insertion succesful. {records_count_in_table} records where inserted!")
    else:
        print("Something went wrong!")
        print(f"Count of provided values: {records_count_in_values}")
        print(f"Count of provided values: {records_count_in_table}")
        conn.rollback()

### Cargar los datos del inventario desde el archivo .json

In [123]:
with open("inventario.json", "r", encoding="utf-8") as file:
    content = json.load(file)   

In [124]:
productos = content["productos"]

In [127]:
productos[:2]

[{'nombre': 'iPhone 15 Pro Max',
  'marca': 'Apple',
  'categoria': 'Smartphones',
  'precio': 1299.99,
  'stock': 15,
  'especificaciones': '6.7 pulgadas, 256GB, Titanio',
  'fecha_ingreso': '2024-01-15'},
 {'nombre': 'Samsung Galaxy S24 Ultra',
  'marca': 'Samsung',
  'categoria': 'Smartphones',
  'precio': 1199.99,
  'stock': 22,
  'especificaciones': '6.8 pulgadas, 512GB, Negro',
  'fecha_ingreso': '2024-01-20'}]

Eliminar el id de cada producto:

In [126]:
productos = [{k:v for k,v in p.items() if k != "id"} for p in productos]

In [70]:
list(productos[0].keys())

['nombre',
 'marca',
 'categoria',
 'precio',
 'stock',
 'especificaciones',
 'fecha_ingreso']

### Insertar datos de inventario de productos

In [128]:
populate_table("productos", ['nombre', 'marca', 'categoria', 'precio',
                              'stock', 'especificaciones', 'fecha_ingreso'], data_structure="dict", values = productos)

Insertion succesful. 50 records where inserted!


## Lectura de datos

In [129]:
def read_table(table, cols="*", filter = None, limit = None, cursor=cursor):
    try:
        if cols != "*":
            columns = ", ".join(cols)
        else:
            columns = "*" 
        query = f'''
        select {columns}
        from {table}
        '''
        if filter:
            query += f"\n\twhere {filter}"
        
        if limit:
            query += f"\nlimit {limit}"
        
        query += " ;"

        return cursor.execute(query).fetchall()
    
    except Exception as e:
        print(f"Error: {e}")

In [131]:
read_table("productos", filter='categoria="Audio"', limit=5)

[(49,
  'Rode NT1 Signature',
  'Rode',
  'Audio',
  449.99,
  14,
  'Micrófono cardioide, 1 pulgada, XLR',
  '2024-02-26'),
 (50,
  'Blue Yeti USB',
  'Blue',
  'Audio',
  99.99,
  72,
  'Micrófono USB, 4 patrones, silenciador',
  '2024-02-27')]

In [133]:
read_table("productos", cols=["nombre", "precio"], limit=3)

[('iPhone 15 Pro Max', 1299.99),
 ('Samsung Galaxy S24 Ultra', 1199.99),
 ('MacBook Pro 16 M3', 2499.99)]

## Actualización de registros

## Eliminación

## Vaciado de la tabla

In [ ]:
def truncate_table(table, conn=conn, cursor=cursor):
    try:
        cursor.execute(f"delete from {table} ;")
        
        # reset the id values
        cursor.execute(f'delete from sqlite_sequence where name ="{table}" ;')

        print(f"Table \"{table}\" has been cleaned from all records.")
        
    except Exception as e:
        print(f"Error: {e}")
    

In [ ]:
truncate_table("productos")

Table "productos" has been cleaned from all records.


## Eliminación de la tabla

In [ ]:
def drop_table(table, cursor=cursor):
    try:
        cursor.execute(f"drop table {table} ;")

        # check if table has been removed from db
        tables_list = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
        tables_list = [i[0] for i in tables_list]
        
        if table not in tables_list:
            print(f"Table \"{table}\" has been removed from the database!")
        else:
            print(f'Table "{table}" could not be removed from database.')
            
    except Exception as e:
        print(f"Error: {e}")